## Setup

In [ ]:
import os
import re
from typing import Sequence
from src.py_src import util
from dotenv import load_dotenv
from tqdm import tqdm
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
load_dotenv()

raw_path = os.getenv("EVENTS_RAW_PATH")
data = {}

begin_year = 1996
end_year = 2024
events_range = range(begin_year, end_year + 1)

## Read Files

### Events Functions

In [ ]:
def find_data_row_events(file_lines_, date_: pd.Timestamp) -> int | None:
    for i, line in enumerate(file_lines_):
        if date_ <= pd.to_datetime('1998-05-08'):
            if  "Reg#" in line.strip():
                return i + 1
        else:
            if "#----------------------------------------------------------" in line.strip():
                return i + 1

    return None

In [ ]:
def find_event_date(file_lines, date_: pd.Timestamp) -> pd.Timestamp | None:
    for i, line in enumerate(file_lines):
        if date_ <= pd.to_datetime('1998-05-08'):
            if f"EDITED EVENTS for {date_.year}" in line.strip():
                date_str = line.strip()[17:]
                return pd.to_datetime(date_str)
        else:
            if f":Date: {date_.year}" in line.strip():
                date_str = line.strip()[6:]
                return pd.to_datetime(date_str)

    return None

In [ ]:
def read_events_lines(path_, date_: pd.Timestamp) -> pd.DataFrame:
    with open(path_, 'r', errors='ignore') as f:
        file_lines = f.readlines()

    data_row = find_data_row_events(file_lines, date_)
    if data_row is None:
        print(f"AVISO: Cabeçalho de Eventos não encontrado em {path_}")
        return pd.DataFrame({'raw_line': []})

    data_lines = [line.rstrip('\n') for line in file_lines[data_row:]]

    df = pd.DataFrame(data_lines, columns=['raw_line'])
    df = df[~df['raw_line'].str.contains("NO EVENT REPORTS", na=False)].copy()
    df = df[df['raw_line'].str.strip() != ''].copy()

    df['date'] = find_event_date(file_lines, date_)

    return df

In [ ]:
def create_timestamps(df_: pd.DataFrame, column_name_: str) -> pd.Series:
    date_str_series = df_['date'].dt.strftime('%Y-%m-%d')
    time_str_series = df_[column_name_]
    full_datetime_str = date_str_series + ' ' + time_str_series

    return pd.to_datetime(full_datetime_str, format='%Y-%m-%d %H%M', errors='coerce')

In [ ]:
def nullify_invalid_time_patterns(df_: pd.DataFrame, columns_names_: Sequence[str], pattern_: re.Pattern[str]) -> pd.DataFrame:
    df_ = df_.copy()

    for column_name in columns_names_:
        mask = df_[column_name].str.match(pattern_, na=False)
        df_.loc[:, column_name] = df_[column_name].where(mask, np.nan)

    return df_

In [ ]:
def format_events(df_raw: pd.DataFrame) -> pd.DataFrame:
    final_cols = ['date','event', 'begin', 'max', 'end', 'obs', 'q', 'type', 'loc_frq', 'particulars', 'reg#']

    if df_raw.empty:
        return pd.DataFrame(columns=final_cols)

    new_data = {
        'date': df_raw['date'],
        'event_num': df_raw['raw_line'].str.slice(0, 5).str.strip(),
        'event_plus': df_raw['raw_line'].str.slice(5, 11).str.strip(),
        'begin': df_raw['raw_line'].str.slice(11, 18).str.strip(),
        'max': df_raw['raw_line'].str.slice(18, 28).str.strip(),
        'end': df_raw['raw_line'].str.slice(28, 34).str.strip(),
        'obs': df_raw['raw_line'].str.slice(34, 39).str.strip(),
        'q': df_raw['raw_line'].str.slice(39, 43).str.strip(),
        'type': df_raw['raw_line'].str.slice(43, 48).str.strip(),
        'loc_frq': df_raw['raw_line'].str.slice(48, 58).str.strip(),
        'particulars': df_raw['raw_line'].str.slice(58, 76).str.strip(),
        'reg#': df_raw['raw_line'].str.slice(76).str.strip()
    }
    df = pd.DataFrame(new_data)

    na_values = ['','////']
    df = df.replace(na_values, np.nan)

    df.loc[:, 'event_plus'] = df['event_plus'].fillna('')
    df.loc[:, 'event'] = (df['event_num'] + df['event_plus']).str.replace(r'[ABU]','',regex=True).str.strip()

    time_columns = ['begin', 'max', 'end']
    pattern = re.compile(r'^\d{4}$')
    df = nullify_invalid_time_patterns(df, time_columns, pattern)

    for column_name in time_columns:
        df.loc[:, column_name] = create_timestamps(df, column_name)

    df.loc[:, 'max'] = np.where(df['max'] < df['begin'], df['max'] + pd.Timedelta(days=1), df['max'])
    df.loc[:, 'end'] = np.where(df['end'] < df['begin'], df['end'] + pd.Timedelta(days=1), df['end'])

    df = df.replace(na_values, np.nan)

    return df[final_cols]

### Main

In [ ]:
for y in tqdm(range(begin_year, end_year+1)):
    data[y] = {}

    year_dir = os.path.join(raw_path, f"{y}")
    if not os.path.isdir(year_dir):
        print(f"ERRO: Diretório não encontrado, pulando ano {y}")
        continue

    events_dir = os.path.join(year_dir,f"{y}_events")
    if not os.path.isdir(events_dir):
        print(f"ERRO: Diretório não encontrado, pulando ano {y}")
        continue

    df_events_list = []
    files_in_dir = set(os.listdir(events_dir))

    for date in pd.date_range(f"{y}-01-01", f"{y}-12-31"):
        m_str = date.strftime("%m")
        d_str = date.strftime("%d")
        file_name = f"{y}{m_str}{d_str}events.txt"


        if file_name not in files_in_dir:
            if date >= pd.to_datetime("1996-07-31"):
                print(f"AVISO : Arquivo não encontrado, pulando {date}")
            continue

        full_path = os.path.join(events_dir, file_name)
        df_day = read_events_lines(full_path, date)
        if df_day is not None and not df_day.empty:
            df_events_list.append(df_day)

    df_events = pd.concat(df_events_list, ignore_index=True)
    df_events = format_events(df_events)
    data[y]['events'] = df_events
    print(f"Success reading {y} events")

## XRA Filtering & Feature Extraction

In [ ]:
df_events_global_list = []
for y in range(begin_year, end_year + 1):
    if 'events' in data[y] and not data[y]['events'].empty:
        df_events_global_list.append(data[y]['events'])

df_events_global = pd.concat(df_events_global_list, ignore_index=True)

df_xra = df_events_global[df_events_global['type'] == 'XRA'].copy()

particulars_split = df_xra['particulars'].str.split(expand=True)

df_xra.loc[:, 'class_expanded'] = particulars_split[0]
df_xra.loc[:, 'flux'] = df_xra['class_expanded'].apply(util.parse_flare_class_expanded)
df_xra.loc[:, 'class'] = df_xra['class_expanded'].str[0].str.upper()
df_xra.loc[:, 'class_numeric'] = df_xra['class'].apply(lambda c: util.flare_class_map.get(c, 0))

df_xra = df_xra[['date', 'reg#', 'class', 'class_numeric', 'flux', 'begin', 'max', 'end']].rename(
    columns={'date': 'ds'})
df_xra = df_xra.dropna(subset=['begin']).copy()

df_xra = df_xra.assign(
    begin=pd.to_datetime(df_xra['begin'], errors='coerce'),
    end=pd.to_datetime(df_xra['end'], errors='coerce'),
    max=pd.to_datetime(df_xra['max'], errors='coerce')
)

df_xra.loc[:, 'duration'] = (df_xra['end'] - df_xra['begin']).dt.total_seconds() / 60
duration_map = df_xra.groupby('class_numeric')['duration'].mean().to_dict()

def impute_end(row):
    if pd.isna(row['end']):
        mean_duration = duration_map.get(row['class_numeric'])
        if pd.isna(mean_duration):
            return row['begin']
        return row['begin'] + pd.to_timedelta(mean_duration, unit='m')
    else:
        return row['end']

df_xra.loc[:, 'end'] = df_xra.apply(impute_end, axis=1)
df_xra = df_xra.drop(columns=['duration'])

### Splitting into Global and By-Region Families

In [ ]:
df_xra = df_xra.copy()

df_xra_global = df_xra.copy().drop(columns=['reg#'])

df_xra_by_region = df_xra.copy()
df_xra_by_region.loc[:, 'reg#'] = df_xra_by_region['reg#'].replace(r'^\s*$', np.nan, regex=True)
df_xra_by_region = df_xra_by_region.dropna(subset=['reg#']).copy()

In [ ]:
df_xra_global

In [ ]:
df_xra_by_region

### Exporting

In [ ]:
treated_global_path = os.getenv("EVENTS_TREATED_GLOBAL_PATH")
treated_by_region_path = os.getenv("EVENTS_TREATED_BY_REGION_PATH")

util.create_dirs(treated_global_path, events_range)
util.create_dirs(treated_by_region_path, events_range)

df_xra_global['year'] = df_xra_global['begin'].dt.year
df_xra_by_region['year'] = df_xra_by_region['begin'].dt.year

print("Exportando DataFrames Global e By-Region...")
for y in tqdm(events_range):
    df_year_global = df_xra_global[df_xra_global['year'] == y].drop(columns=['year'])
    if not df_year_global.empty:
        year_dir_global = os.path.join(treated_global_path, str(y))
        df_year_global.to_parquet(os.path.join(year_dir_global, f"{y}_xra_events_global.parquet"), index=False)

    df_year_region = df_xra_by_region[df_xra_by_region['year'] == y].drop(columns=['year'])
    if not df_year_region.empty:
        year_dir_region = os.path.join(treated_by_region_path, str(y))
        df_year_region.to_parquet(os.path.join(year_dir_region, f"{y}_xra_events_by_region.parquet"), index=False)

df_xra_global = df_xra_global.drop(columns=['year'], errors='ignore')
df_xra_by_region = df_xra_by_region.drop(columns=['year'], errors='ignore')

print("Exportação concluída!")